In [1]:
import os
import numpy as np
import pandas as pd


from models import BirdModel
from losses import CASL, ASL, BCE
from metrics import AUC, MAP, CMAP
from utils import Lab, make_teachers
from training.baseline import Trainer as Baseline
from training.semi_supervised import Trainer as SSL
from training.xc_pretraining import Trainer as Pretraining

/home/antoine/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
config = {
    'seed':2,
    'batch_size':32,
    "backbone":"tf_efficientnetv2_b0",
    "loss":BCE(),
    'mel':{'n_mels':160, 'f_min':50, 'n_fft':2048, 'hop_length':512}, 
    "mix":{"alpha":2, "theta":.1},
    "pretrained":True,
    "model":BirdModel,
    'pretrained':'imagenet',
    'train_only':False,
    'backbone_pooling':'avg', 
    'dropout':0.3,
    'sampling':'none',
    'duration':5,
}

configs = [
    {'backbone':'tf_efficientnetv2_s'},]

for n_epochs in [16]:
    for cfg in configs:
        trainer = Pretraining(config=config|cfg, fold=0)
        model = trainer.train(epochs=n_epochs)
        torch.save(trainer.model.backbone.state_dict(), f"pretrained_models/XC_{n_epochs}_{(config|cfg)['backbone']}_{(config|cfg)['duration']}s.pth")

In [2]:
config = {
    'seed':2,
    'batch_size':32,
    "backbone":"tf_efficientnetv2_b0",
    "loss":CASL(),
    'mel':{'n_mels':224, 'f_min':50, 'n_fft':2048, 'hop_length':512}, 
    "mix":{"alpha":2, "theta":.1},
    "pretrained":'xc',
    "model":BirdModel,
    'train_only':False,
    'extra_data':False,
    'soundscape_data':True,
    'synonymous_taxa':True,
    'sampling_gamma':0.8,
    'backbone_pooling':'avg', 
    'dropout':0.3,
    'augmentation':'light',
    'sampling':'none'
}

delivery_id = 'Baseline - CASL - efficientnetv2_b0'
lab = Lab(delivery_id)
for fold in range(5):
    trainer = Baseline(config=config|{'data_version':'processed', "loss":CASL()}, fold=fold)
    model = trainer.train(epochs=20, Lab=lab)
    lab.ship()

make_teachers([delivery_id], 1)

In [3]:
config = {
    'seed':8,
    'batch_size':20,
    "backbone":"tf_efficientnetv2_b0",
    "teacher_backbone":"tf_efficientnetv2_b0",
    "student_init":"none",
    "loss":CASL(),
    'mel':{'n_mels':224, 'f_min':50, 'n_fft':2048, 'hop_length':512}, 
    "mix":{"alpha":1, "theta":.1},
    'augmentation':'extreme',
    "pretrained":True,
    "model":BirdModel,
    'train_only':True,
    'extra_data':False,
    'soundscape_data':True,
    'synonymous_taxa':True,
    'sampling_gamma':0.8,
    'backbone_pooling':'avg', 
    'dropout':0.3,
    'sampling':'none',
    'teacher_version':1,
    'ema':0,
    'T':0.5,
}

lab = Lab('teacher - CASL - r1')
for fold in [0]:
    trainer = SSL(config=config|{'round':1, 'student_init':'teacher', 'data_version':'processed'}, fold=fold)
    model = trainer.train(epochs=32, Lab=lab)
    lab.ship()